In [1]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import copy
import os
import time
import sys
import pandas as pd
from sklearn.preprocessing import StandardScaler
import seaborn as sns    
from mpl_toolkits.mplot3d import Axes3D
from sklearn.model_selection import train_test_split



In [11]:

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def lstm_cell_forward(xt, a_prev, c_prev, parameters):
    """
    Arguments:
    xt -- Données d'entrée à time-step t, array de forme (n_x, m)
    a_prev -- Etat caché précédent, array de forme (n_a, m)
    c_prev -- Etat de la cellule précédente, array de forme (n_a, m)
    parameters -- Dictionnaire Python contenant:
                Wf -- Poids de la forget gate, array de forme (n_a, n_a + n_x)
                bf -- Biais de la forget gate, array de forme (n_a, 1)
                Wi -- Poids de l'update gate, array de forme (n_a, n_a + n_x)
                bi -- Biais de l'update gate, array de forme (n_a, 1)
                Wc -- Poids de la première "tanh", array de forme (n_a, n_a + n_x)
                bc -- Biais de la première "tanh", array de forme (n_a, 1)
                Wo -- Poids de l'output gate, array de forme (n_a, n_a + n_x)
                bo -- Biais de l'output gate, array de forme (n_a, 1)
                Wy -- Poids pour l'état caché, array de forme (n_y, n_a)
                by -- Biais pour l'état caché, array de forme (n_y, 1)

    Returns:
    a_next -- Prochain état caché, array de forme (n_a, m)
    c_next -- Prochain état de cellule, array de forme (n_a, m)
    yt_pred -- Prédiction à time-step t, array de forme (n_y, m)
    cache -- Tuple de valeurs pour la backpropagation
    """

    # Récupérer les paramètres du dictionnaire
    Wf = parameters["Wf"]
    bf = parameters["bf"]
    Wi = parameters["Wi"]
    bi = parameters["bi"]
    Wc = parameters["Wc"]
    bc = parameters["bc"]
    Wo = parameters["Wo"]
    bo = parameters["bo"]
    Wy = parameters["Wy"]
    by = parameters["by"]

    # Récupérer les dimensions
    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    # Concaténer a_prev et xt
    concat = np.zeros((n_a + n_x, m))
    concat[: n_a, :] = a_prev
    concat[n_a:, :] = xt

    # Calculer les valeurs pour ft, it, cct, c_next, ot, a_next
    ft = sigmoid(np.matmul(Wf, concat) + bf)
    it = sigmoid(np.matmul(Wi, concat) + bi)
    cct = np.tanh(np.matmul(Wc, concat) + bc)
    c_next = (ft * c_prev) + (it * cct)
    ot = sigmoid(np.matmul(Wo, concat) + bo)
    a_next = ot * np.tanh(c_next)

    # Calculer la prédiction
    yt_pred = softmax(np.matmul(Wy, a_next) + by)

    # Stocker les valeurs pour la backpropagation
    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return a_next, c_next, yt_pred, cache

def lstm_cell_backward(da_next, dc_next, cache):
    """
    Arguments:
    da_next -- Gradient du prochain état caché, array de forme (n_a, m)
    dc_next -- Gradient du prochain état de cellule, array de forme (n_a, m)
    cache -- Cache du forward pass

    Returns:
    gradients -- Dictionnaire contenant les gradients
    """

    # Récupérer les informations du cache
    (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters) = cache

    # Récupérer les dimensions
    n_x, m = xt.shape
    n_a, m = a_next.shape

    # Calculer les dérivées des portes
    dot = da_next * np.tanh(c_next) * ot * (1 - ot)
    dcct = (dc_next * it + ot * (1 - np.square(np.tanh(c_next))) * it * da_next) * (1 - np.square(cct))
    dit = (dc_next * cct + ot * (1 - np.square(np.tanh(c_next))) * cct * da_next) * it * (1 - it)
    dft = (dc_next * c_prev + ot * (1 - np.square(np.tanh(c_next))) * c_prev * da_next) * ft * (1 - ft)

    concat = np.concatenate((a_prev, xt), axis=0)

    # Calculer les dérivées des paramètres
    dWf = np.dot(dft, concat.T)
    dWi = np.dot(dit, concat.T)
    dWc = np.dot(dcct, concat.T)
    dWo = np.dot(dot, concat.T)
    dbf = np.sum(dft, axis=1, keepdims=True)
    dbi = np.sum(dit, axis=1, keepdims=True)
    dbc = np.sum(dcct, axis=1, keepdims=True)
    dbo = np.sum(dot, axis=1, keepdims=True)

    # Calculer les dérivées par rapport à l'état caché précédent, l'état de cellule précédent et l'entrée
    da_prev = np.dot(parameters['Wf'][:, :n_a].T, dft) + np.dot(parameters['Wi'][:, :n_a].T, dit) + np.dot(
        parameters['Wc'][:, :n_a].T, dcct) + np.dot(parameters['Wo'][:, :n_a].T, dot)
    dc_prev = dc_next * ft + ot * (1 - np.square(np.tanh(c_next))) * ft * da_next
    dxt = np.dot(parameters['Wf'][:, n_a:].T, dft) + np.dot(parameters['Wi'][:, n_a:].T, dit) + np.dot(
        parameters['Wc'][:, n_a:].T, dcct) + np.dot(parameters['Wo'][:, n_a:].T, dot)

    # Sauvegarder les gradients
    gradients = {"dxt": dxt, "da_prev": da_prev, "dc_prev": dc_prev, "dWf": dWf, "dbf": dbf, "dWi": dWi, "dbi": dbi,
                 "dWc": dWc, "dbc": dbc, "dWo": dWo, "dbo": dbo}

    return gradients

def lstm_forward(x, a0, parameters):
    """
    Arguments:
    x -- Données d'entrée pour chaque time-step, array de forme (n_x, m, T_x)
    a0 -- État caché initial, array de forme (n_a, m)
    parameters -- Dictionnaire Python des paramètres LSTM

    Returns:
    a -- États cachés pour chaque time-step, array de forme (n_a, m, T_x)
    y -- Prédictions pour chaque time-step, array de forme (n_y, m, T_x)
    c -- États de cellule pour chaque time-step, array de forme (n_a, m, T_x)
    caches -- Tuple de valeurs pour la backpropagation
    """

    # Initialiser les caches
    caches = []

    # Récupérer les dimensions
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wy"].shape

    # Initialiser a, c et y avec des zéros
    a = np.zeros((n_a, m, T_x))
    c = a.copy()
    y = np.zeros((n_y, m, T_x))

    # Initialiser a_next et c_next
    a_next = a0
    c_next = np.zeros(a_next.shape)

    # Boucle sur tous les time-steps
    for t in range(T_x):
        # Mettre à jour a_next, c_next, calculer la prédiction et obtenir le cache
        a_next, c_next, yt, cache = lstm_cell_forward(x[:, :, t], a_next, c_next, parameters)
        # Sauvegarder les valeurs
        a[:, :, t] = a_next
        y[:, :, t] = yt
        c[:, :, t] = c_next
        # Ajouter le cache
        caches.append(cache)

    # Stocker les valeurs pour la backpropagation
    caches = (caches, x)

    return a, y, c, caches

def lstm_backward(da, caches):
    """
    Arguments:
    da -- Gradient par rapport aux états cachés, array de forme (n_a, m, T_x)
    caches -- Cache du forward pass

    Returns:
    gradients -- Dictionnaire Python contenant les gradients
    """

    # Récupérer les valeurs du cache
    (caches, x) = caches
    (a1, c1, a0, c0, f1, i1, cc1, o1, x1, parameters) = caches[0]

    # Récupérer les dimensions
    n_a, m, T_x = da.shape
    n_x, m = x1.shape

    # Initialiser les gradients
    dx = np.zeros((n_x, m, T_x))
    da0 = np.zeros((n_a, m))
    da_prevt = np.zeros(da0.shape)
    dc_prevt = np.zeros(da0.shape)
    dWf = np.zeros((n_a, n_a + n_x))
    dWi = np.zeros(dWf.shape)
    dWc = np.zeros(dWf.shape)
    dWo = np.zeros(dWf.shape)
    dbf = np.zeros((n_a, 1))
    dbi = np.zeros(dbf.shape)
    dbc = np.zeros(dbf.shape)
    dbo = np.zeros(dbf.shape)

    # Boucle sur la séquence en sens inverse
    for t in reversed(range(T_x)):
        # Calculer tous les gradients à l'aide de lstm_cell_backward
        gradients = lstm_cell_backward(da[:, :, t] + da_prevt, dc_prevt, caches[t])
        # Stocker ou ajouter les gradients aux gradients de l'étape précédente
        dx[:, :, t] = gradients["dxt"]
        dWf += gradients["dWf"]
        dWi += gradients["dWi"]
        dWc += gradients["dWc"]
        dWo += gradients["dWo"]
        dbf += gradients["dbf"]
        dbi += gradients["dbi"]
        dbc += gradients["dbc"]
        dbo += gradients["dbo"]
        da_prevt = gradients["da_prev"]
        dc_prevt = gradients["dc_prev"]

    # Définir le premier gradient d'activation
    da0 = gradients["da_prev"]

    # Stocker les gradients dans un dictionnaire Python
    gradients = {"dx": dx, "da0": da0, "dWf": dWf, "dbf": dbf, "dWi": dWi, "dbi": dbi,
                 "dWc": dWc, "dbc": dbc, "dWo": dWo, "dbo": dbo}

    return gradients

def initialize_adam_for_lstm(parameters):
    """
    Initialise v et s pour les paramètres du LSTM.

    Arguments:
    parameters -- Dictionnaire Python contenant les paramètres du LSTM.

    Returns:
    v -- Dictionnaire Python qui contiendra la moyenne mobile exponentielle du gradient.
    s -- Dictionnaire Python qui contiendra la moyenne mobile exponentielle du carré du gradient.
    """
    v = {}
    s = {}

    # Initialiser v, s pour tous les paramètres du LSTM
    for key in parameters.keys():
        v["d" + key] = np.zeros_like(parameters[key])
        s["d" + key] = np.zeros_like(parameters[key])

    return v, s

def update_parameters_with_adam_for_lstm(parameters, grads, v, s, t, learning_rate=0.01,
                                         beta1=0.9, beta2=0.999, epsilon=1e-8):

    v_corrected = {}  # Estimation du premier moment corrigée du biais
    s_corrected = {}  # Estimation du second moment corrigée du biais

    # Effectuer la mise à jour Adam sur tous les paramètres
    for key in parameters.keys():
        # Clé correspondante dans les dictionnaires grads, v, s
        d_key = "d" + key

        # S'assurer que nous avons le gradient correspondant
        if d_key not in grads:
            continue

        # Moyenne mobile des gradients
        v[d_key] = beta1 * v[d_key] + (1 - beta1) * grads[d_key]

        # Calcul de l'estimation du premier moment corrigée du biais
        v_corrected[d_key] = v[d_key] / (1 - beta1**t)

        # Moyenne mobile des carrés des gradients
        s[d_key] = beta2 * s[d_key] + (1 - beta2) * (grads[d_key]**2)

        # Calcul de l'estimation du second moment corrigée du biais
        s_corrected[d_key] = s[d_key] / (1 - beta2**t)

        # Mise à jour des paramètres
        parameters[key] = parameters[key] - learning_rate * v_corrected[d_key] / (np.sqrt(s_corrected[d_key]) + epsilon)

    return parameters, v, s

def initialize_lstm_parameters(n_a, n_x, n_y):
    """
    Initialise les paramètres du LSTM.

    Arguments:
    n_a -- nombre d'unités dans la couche cachée
    n_x -- taille d'entrée
    n_y -- taille de sortie

    Returns:
    parameters -- dictionnaire Python contenant les paramètres initialisés
    """
    np.random.seed(1)

    # Initialisation avec He/Xavier
    Wf = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bf = np.zeros((n_a, 1))
    Wi = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bi = np.zeros((n_a, 1))
    Wc = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bc = np.zeros((n_a, 1))
    Wo = np.random.randn(n_a, n_a + n_x) * np.sqrt(1. / (n_a + n_x))
    bo = np.zeros((n_a, 1))
    Wy = np.random.randn(n_y, n_a) * np.sqrt(1. / n_a)
    by = np.zeros((n_y, 1))

    parameters = {"Wf": Wf, "bf": bf, "Wi": Wi, "bi": bi, "Wc": Wc, "bc": bc, "Wo": Wo, "bo": bo, "Wy": Wy, "by": by}

    return parameters

def train_lstm(X_train, Y_train, n_a, n_x, n_y, num_epochs=10, seed=1, learning_rate=0.01, initial_params=None):
    """
    Entraîne un LSTM sur les données fournies, avec possibilité d'initialiser avec des paramètres existants.

    Arguments:
    X_train -- données d'entrée, numpy array de forme (n_x, m, T_x)
    Y_train -- étiquettes, numpy array de forme (n_y, m, T_x)
    n_a -- nombre d'unités dans la couche cachée
    n_x -- taille d'entrée
    n_y -- taille de sortie
    num_epochs -- nombre d'époques d'entraînement
    seed -- graine pour la reproductibilité
    learning_rate -- taux d'apprentissage
    initial_params -- paramètres initiaux (optionnel)

    Returns:
    parameters -- paramètres finaux
    parameters_history -- historique des paramètres à chaque époque
    loss_history -- historique des pertes
    """
    np.random.seed(seed)

    # Initialisation des paramètres
    if initial_params is None:
        parameters = initialize_lstm_parameters(n_a, n_x, n_y)
    else:
        parameters = copy.deepcopy(initial_params)

    parameters_history = []
    loss_history = []

    # Initialiser Adam
    v, s = initialize_adam_for_lstm(parameters)
    t = 0  # Compteur pour Adam

    for epoch in range(num_epochs):
        print(f"Époque {epoch+1}/{num_epochs}")

        # Forward pass
        a0 = np.zeros((n_a, X_train.shape[1]))
        a, y_pred, c, caches = lstm_forward(X_train, a0, parameters)

        # Calcul de la perte (cross-entropy)
        loss = -np.sum(Y_train * np.log(y_pred + 1e-8)) / (Y_train.shape[1] * Y_train.shape[2])
        loss_history.append(loss)
        print(f"Loss: {loss:.4f}")

        # Initialisation du gradient de sortie
        da = np.zeros_like(a)

        # Créer un dictionnaire complet pour les gradients
        gradients = {}

        # Pour chaque pas de temps, calculer le gradient
        dWy = np.zeros_like(parameters["Wy"])
        dby = np.zeros_like(parameters["by"])

        for t_idx in range(Y_train.shape[2]):
            # Gradient de la cross-entropy
            dy = y_pred[:, :, t_idx] - Y_train[:, :, t_idx]
            # Accumuler les gradients pour Wy et by
            dWy += np.dot(dy, a[:, :, t_idx].T)
            dby += np.sum(dy, axis=1, keepdims=True)
            # Gradient par rapport à a
            da[:, :, t_idx] = np.dot(parameters["Wy"].T, dy)

        # Backward pass pour le reste des paramètres LSTM
        lstm_gradients = lstm_backward(da, caches)

        # Combiner tous les gradients
        gradients = lstm_gradients.copy()
        gradients["dWy"] = dWy
        gradients["dby"] = dby

        # Mise à jour des paramètres avec Adam
        t += 1
        parameters, v, s = update_parameters_with_adam_for_lstm(parameters, gradients, v, s, t, learning_rate)

        # Sauvegarde des paramètres après cette époque
        parameters_history.append(copy.deepcopy(parameters))

    return parameters, parameters_history, loss_history

In [12]:
# Fonctions corrigées pour la régression

def lstm_cell_forward_regression(xt, a_prev, c_prev, parameters):
    """
    LSTM cell forward pour régression (sans softmax).
    """
    # Récupérer les paramètres du dictionnaire
    Wf = parameters["Wf"]
    bf = parameters["bf"]
    Wi = parameters["Wi"]
    bi = parameters["bi"]
    Wc = parameters["Wc"]
    bc = parameters["bc"]
    Wo = parameters["Wo"]
    bo = parameters["bo"]
    Wy = parameters["Wy"]
    by = parameters["by"]

    # Récupérer les dimensions
    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    # Concaténer a_prev et xt
    concat = np.zeros((n_a + n_x, m))
    concat[: n_a, :] = a_prev
    concat[n_a:, :] = xt

    # Calculer les valeurs pour ft, it, cct, c_next, ot, a_next
    ft = sigmoid(np.matmul(Wf, concat) + bf)
    it = sigmoid(np.matmul(Wi, concat) + bi)
    cct = np.tanh(np.matmul(Wc, concat) + bc)
    c_next = (ft * c_prev) + (it * cct)
    ot = sigmoid(np.matmul(Wo, concat) + bo)
    a_next = ot * np.tanh(c_next)

    # Calculer la prédiction (régression linéaire, pas de softmax)
    yt_pred = np.matmul(Wy, a_next) + by

    # Stocker les valeurs pour la backpropagation
    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return a_next, c_next, yt_pred, cache

def lstm_forward_regression(x, a0, parameters):
    """
    LSTM forward pour régression.
    """
    # Initialiser les caches
    caches = []

    # Récupérer les dimensions
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wy"].shape

    # Initialiser a, c et y avec des zéros
    a = np.zeros((n_a, m, T_x))
    c = a.copy()
    y = np.zeros((n_y, m, T_x))

    # Initialiser a_next et c_next
    a_next = a0
    c_next = np.zeros(a_next.shape)

    # Boucle sur tous les time-steps
    for t in range(T_x):
        # Mettre à jour a_next, c_next, calculer la prédiction et obtenir le cache
        a_next, c_next, yt, cache = lstm_cell_forward_regression(x[:, :, t], a_next, c_next, parameters)
        # Sauvegarder les valeurs
        a[:, :, t] = a_next
        y[:, :, t] = yt
        c[:, :, t] = c_next
        # Ajouter le cache
        caches.append(cache)

    # Stocker les valeurs pour la backpropagation
    caches = (caches, x)

    return a, y, c, caches

def train_lstm_regression(X_train, Y_train, n_a, n_x, n_y, num_epochs=10, seed=1, learning_rate=0.01, initial_params=None):
    """
    Entraîne un LSTM pour la régression.
    """
    np.random.seed(seed)

    # Initialisation des paramètres
    if initial_params is None:
        parameters = initialize_lstm_parameters(n_a, n_x, n_y)
    else:
        parameters = copy.deepcopy(initial_params)

    parameters_history = []
    loss_history = []

    # Initialiser Adam
    v, s = initialize_adam_for_lstm(parameters)
    t = 0  # Compteur pour Adam

    for epoch in range(num_epochs):
        print(f"Époque {epoch+1}/{num_epochs}")

        # Forward pass
        a0 = np.zeros((n_a, X_train.shape[1]))
        a, y_pred, c, caches = lstm_forward_regression(X_train, a0, parameters)

        # Calcul de la perte MSE pour régression
        loss = np.mean((Y_train - y_pred) ** 2)
        loss_history.append(loss)
        print(f"MSE Loss: {loss:.6f}")

        # Initialisation du gradient de sortie
        da = np.zeros_like(a)

        # Créer un dictionnaire complet pour les gradients
        gradients = {}

        # Pour chaque pas de temps, calculer le gradient
        dWy = np.zeros_like(parameters["Wy"])
        dby = np.zeros_like(parameters["by"])

        for t_idx in range(Y_train.shape[2]):
            # Gradient MSE
            dy = 2 * (y_pred[:, :, t_idx] - Y_train[:, :, t_idx]) / (Y_train.shape[1] * Y_train.shape[2])
            # Accumuler les gradients pour Wy et by
            dWy += np.dot(dy, a[:, :, t_idx].T)
            dby += np.sum(dy, axis=1, keepdims=True)
            # Gradient par rapport à a
            da[:, :, t_idx] = np.dot(parameters["Wy"].T, dy)

        # Backward pass pour le reste des paramètres LSTM
        lstm_gradients = lstm_backward(da, caches)

        # Combiner tous les gradients
        gradients = lstm_gradients.copy()
        gradients["dWy"] = dWy
        gradients["dby"] = dby

        # Mise à jour des paramètres avec Adam
        t += 1
        parameters, v, s = update_parameters_with_adam_for_lstm(parameters, gradients, v, s, t, learning_rate)

        # Sauvegarde des paramètres après cette époque
        parameters_history.append(copy.deepcopy(parameters))

    return parameters, parameters_history, loss_history

In [13]:

def load_jena_climate_data(file_path=None, n_clients=3, n_seeds_per_client=1, n_x=8, n_y=1, 
                          sequence_length=10, test_size=0.2, target_column='T (degC)'):
    """
    Charge et prépare le Jena Climate Dataset pour l'apprentissage fédéré LSTM.
    """
    print("Chargement du Jena Climate Dataset")
    
    # Recherche automatique du fichier
    if file_path is None:
        possible_paths = [
            "./jena_climate_2009_2016.csv",
            "./data/jena_climate_2009_2016.csv",
            "./datasets/jena_climate_2009_2016.csv",
            "jena_climate_2009_2016.csv"
        ]
        
        for path in possible_paths:
            if os.path.exists(path):
                file_path = path
                break
    
    if file_path is None or not os.path.exists(file_path):
        raise FileNotFoundError(f"Dataset non trouvé. Téléchargez jena_climate_2009_2016.csv depuis Kaggle")
    
    # Chargement et nettoyage
    df = pd.read_csv(file_path)
    print(f"Données brutes: {df.shape}")
    
    df_clean = clean_jena_data(df, target_column)
    feature_columns = select_jena_features(df_clean, n_x, target_column)
    
    clients_data = create_federated_splits(df_clean, feature_columns, target_column, 
                                         n_clients, n_seeds_per_client, sequence_length)
    
    return clients_data

def clean_jena_data(df, target_column='T (degC)'):
    """Nettoie et prépare les données."""
    df_clean = df.copy()
    
    # Conversion de date si présente
    if 'Date Time' in df_clean.columns:
        df_clean['Date Time'] = pd.to_datetime(df_clean['Date Time'], format='%d.%m.%Y %H:%M:%S')
        df_clean = df_clean.set_index('Date Time')
    
    # Interpolation des valeurs manquantes
    numeric_columns = df_clean.select_dtypes(include=[np.number]).columns
    df_clean[numeric_columns] = df_clean[numeric_columns].interpolate(method='linear')
    df_clean = df_clean.dropna()
    
    # Vérification de la colonne cible
    if target_column not in df_clean.columns:
        numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            target_column = numeric_cols[0]
            print(f"Colonne cible changée pour: {target_column}")
    
    print(f"Données nettoyées: {df_clean.shape}")
    return df_clean

def select_jena_features(df, n_x, target_column):
    """Sélectionne les meilleures features."""
    priority_features = [
        'p (mbar)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)',
        'wd (deg)', 'rh (%)', 'Tdew (degC)', 'VPmax (mbar)',
        'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)', 'H2OC (mmol/mol)'
    ]
    
    # Features disponibles par priorité
    available_features = [f for f in priority_features if f in df.columns and f != target_column]
    
    # Ajouter d'autres colonnes numériques si nécessaire
    if len(available_features) < n_x:
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        for col in numeric_cols:
            if col not in available_features and col != target_column:
                available_features.append(col)
                if len(available_features) >= n_x:
                    break
    
    selected_features = available_features[:n_x]
    
    if len(selected_features) < n_x:
        raise ValueError(f"Features insuffisantes: {len(selected_features)} < {n_x}")
    
    print(f"Features sélectionnées: {selected_features}")
    return selected_features

def create_federated_splits(df, feature_columns, target_column, n_clients, n_seeds_per_client, sequence_length, max_samples_per_client=5000):
    """Crée des divisions fédérées temporelles."""
    print(f"Création de {n_clients} clients avec {n_seeds_per_client} seeds chacun")
    
    total_samples = len(df)
    samples_per_client = min(total_samples // n_clients, max_samples_per_client)
    clients_data = []
    
    for client_id in range(n_clients):
        start_idx = client_id * samples_per_client
        end_idx = min(start_idx + samples_per_client, total_samples)
        
        client_df = df.iloc[start_idx:end_idx].copy()
        print(f"Client {client_id}: {len(client_df)} échantillons")
        
        # Extraction et normalisation
        X_client = client_df[feature_columns].values
        y_client = client_df[target_column].values.reshape(-1, 1)
        
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        X_normalized = scaler_X.fit_transform(X_client)
        y_normalized = scaler_y.fit_transform(y_client)
        
        client_seeds_data = []
        
        # Création des seeds
        for seed_id in range(n_seeds_per_client):
            if n_seeds_per_client > 1:
                seed_size = len(X_normalized) // n_seeds_per_client
                seed_start = seed_id * seed_size
                seed_end = (seed_id + 1) * seed_size if seed_id < n_seeds_per_client - 1 else len(X_normalized)
                X_seed = X_normalized[seed_start:seed_end]
                y_seed = y_normalized[seed_start:seed_end]
            else:
                X_seed = X_normalized
                y_seed = y_normalized
            
            # Création des séquences
            X_sequences, y_sequences = create_sequences_jena(X_seed, y_seed, sequence_length)
            
            if len(X_sequences) > 0:
                # Limiter le nombre de séquences pour éviter les trop gros batch
                max_sequences = min(len(X_sequences), 2000)
                X_sequences = X_sequences[:max_sequences]
                y_sequences = y_sequences[:max_sequences]
                
                # Format LSTM (n_x, batch_size, sequence_length)
                X_lstm = np.transpose(X_sequences, (2, 0, 1))
                # Pour y_sequences: (n_samples, n_y) -> (n_y, n_samples)
                y_lstm = np.transpose(y_sequences, (2, 0, 1))
                
                print(f"  Seed {seed_id}: {X_lstm.shape} -> {y_lstm.shape}")
                client_seeds_data.append((X_lstm, y_lstm))
        
        if client_seeds_data:
            clients_data.append(client_seeds_data)
    
    print(f"{len(clients_data)} clients créés")
    return clients_data

def create_sequences_jena(X, y, sequence_length):
    """Crée des séquences temporelles."""
    if len(X) < sequence_length:
        return np.array([]), np.array([])
    
    X_sequences = []
    y_sequences = []
    
    for i in range(len(X) - sequence_length):
        X_sequences.append(X[i:i + sequence_length])
        # S'assurer que y garde sa forme 2D
        y_seq = y[i + sequence_length]
        if len(y_seq.shape) == 1:
            y_seq = y_seq.reshape(-1, 1)
        y_sequences.append(y_seq)
    
    return np.array(X_sequences), np.array(y_sequences)

def create_jena_test_dataset(clients_data, batch_size, n_x, n_y, sequence_length):
    """Crée un dataset de test à partir des données clients."""
    print("Création du dataset de test")
    
    test_samples_X = []
    test_samples_Y = []
    
    for client_seeds_data in clients_data:
        if len(client_seeds_data) > 0:
            X, Y = client_seeds_data[0]
            n_samples = X.shape[1]
            n_test = min(batch_size // len(clients_data), n_samples // 3)
            
            if n_test > 0:
                test_samples_X.append(X[:, :n_test, :])
                test_samples_Y.append(Y[:, :n_test, :])
    
    if test_samples_X:
        X_test = np.concatenate(test_samples_X, axis=1)
        Y_test = np.concatenate(test_samples_Y, axis=1)
        
        # Ajustement de la taille du batch
        if X_test.shape[1] > batch_size:
            X_test = X_test[:, :batch_size, :]
            Y_test = Y_test[:, :batch_size, :]
        
        print(f"Dataset de test: {X_test.shape} -> {Y_test.shape}")
        return X_test, Y_test
    else:
        raise ValueError("Impossible de créer le dataset de test")

In [14]:

def compute_rmse(y_true, y_pred):
    """
    Calcule le RMSE entre les valeurs vraies et prédites.
    
    Arguments:
    y_true -- valeurs vraies, array de forme (n_y, m, T_x)
    y_pred -- valeurs prédites, array de forme (n_y, m, T_x)
    
    Returns:
    rmse -- Root Mean Square Error
    """
    mse = np.mean((y_true - y_pred) ** 2)
    rmse = np.sqrt(mse)
    return rmse

def compute_loss(y_true, y_pred):
    """
    Calcule la loss cross-entropy entre les valeurs vraies et prédites.
    
    Arguments:
    y_true -- valeurs vraies, array de forme (n_y, m, T_x)
    y_pred -- valeurs prédites, array de forme (n_y, m, T_x)
    
    Returns:
    loss -- Cross-entropy loss
    """
    # Éviter log(0) en ajoutant une petite valeur epsilon
    epsilon = 1e-8
    y_pred_clipped = np.clip(y_pred, epsilon, 1 - epsilon)
    
    # Calcul de la cross-entropy
    loss = -np.sum(y_true * np.log(y_pred_clipped)) / (y_true.shape[1] * y_true.shape[2])
    return loss

def evaluate_lstm_clients(clients_data, parameters_list, n_a, n_x, n_y):
    """
    Évalue les performances LSTM pour chaque client.
    
    Arguments:
    clients_data -- données des clients
    parameters_list -- liste des paramètres entraînés pour chaque client
    n_a -- nombre d'unités cachées
    n_x -- taille d'entrée
    n_y -- taille de sortie
    
    Returns:
    client_metrics -- dictionnaire contenant RMSE et loss pour chaque client
    """
    client_metrics = {
        'client_ids': [],
        'rmse_scores': [],
        'loss_scores': []
    }
    
    print("Évaluation des performances par client:")
    
    for client_id, (client_seeds_data, client_params) in enumerate(zip(clients_data, parameters_list)):
        if len(client_seeds_data) > 0:
            # Prendre la première seed pour l'évaluation
            X_client, Y_client = client_seeds_data[0]
            
            # Forward pass avec les paramètres du client
            a0 = np.zeros((n_a, X_client.shape[1]))
            a, y_pred, c, caches = lstm_forward(X_client, a0, client_params)
            
            # Calcul des métriques
            rmse = compute_rmse(Y_client, y_pred)
            loss = compute_loss(Y_client, y_pred)
            
            # Stockage des résultats
            client_metrics['client_ids'].append(client_id)
            client_metrics['rmse_scores'].append(rmse)
            client_metrics['loss_scores'].append(loss)
            
            print(f"Client {client_id}: RMSE = {rmse:.4f}, Loss = {loss:.4f}")
    
    return client_metrics

def plot_client_performance(client_metrics, save_path=None):
    """
    Visualise les performances RMSE et Loss pour chaque client.
    
    Arguments:
    client_metrics -- dictionnaire contenant les métriques des clients
    save_path -- chemin pour sauvegarder le graphique (optionnel)
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    client_ids = client_metrics['client_ids']
    rmse_scores = client_metrics['rmse_scores']
    loss_scores = client_metrics['loss_scores']
    
    # Graphique RMSE
    bars1 = ax1.bar(client_ids, rmse_scores, color='skyblue', alpha=0.7, edgecolor='navy')
    ax1.set_xlabel('Client ID')
    ax1.set_ylabel('RMSE')
    ax1.set_title('RMSE par Client')
    ax1.grid(True, alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for bar, value in zip(bars1, rmse_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{value:.4f}', ha='center', va='bottom', fontsize=10)
    
    # Graphique Loss
    bars2 = ax2.bar(client_ids, loss_scores, color='lightcoral', alpha=0.7, edgecolor='darkred')
    ax2.set_xlabel('Client ID')
    ax2.set_ylabel('Cross-Entropy Loss')
    ax2.set_title('Loss par Client')
    ax2.grid(True, alpha=0.3)
    
    # Ajouter les valeurs sur les barres
    for bar, value in zip(bars2, loss_scores):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{value:.4f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Graphique sauvegardé: {save_path}")
    
    plt.show()
    
    # Affichage des statistiques
    print(f"\nStatistiques globales:")
    print(f"RMSE moyen: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
    print(f"Loss moyenne: {np.mean(loss_scores):.4f} ± {np.std(loss_scores):.4f}")
    print(f"Meilleur client (RMSE): Client {client_ids[np.argmin(rmse_scores)]} ({min(rmse_scores):.4f})")
    print(f"Meilleur client (Loss): Client {client_ids[np.argmin(loss_scores)]} ({min(loss_scores):.4f})")

def evaluate_and_plot_clients(clients_data, parameters_list, n_a, n_x, n_y, save_path=None):
    """
    Fonction combinée pour évaluer et visualiser les performances des clients.
    
    Arguments:
    clients_data -- données des clients
    parameters_list -- liste des paramètres entraînés pour chaque client
    n_a -- nombre d'unités cachées
    n_x -- taille d'entrée
    n_y -- taille de sortie
    save_path -- chemin pour sauvegarder le graphique (optionnel)
    
    Returns:
    client_metrics -- dictionnaire contenant les métriques
    """
    # Évaluation
    client_metrics = evaluate_lstm_clients(clients_data, parameters_list, n_a, n_x, n_y)
    
    # Visualisation
    plot_client_performance(client_metrics, save_path)
    
    return client_metrics

In [15]:
def main():
    """
    Fonction principale pour exécuter l'entraînement LSTM fédéré sur le dataset Jena Climate.
    """
    print("=" * 60)
    print("LSTM FÉDÉRÉ - JENA CLIMATE DATASET")
    print("=" * 60)
    
    # Configuration des paramètres
    n_clients = 3
    n_seeds_per_client = 1
    n_x = 8  # Nombre de features d'entrée
    n_y = 1  # Dimension de sortie (régression)
    n_a = 32  # Nombre d'unités cachées (réduit pour la vitesse)
    sequence_length = 10
    num_epochs = 10  # Réduit pour les tests
    learning_rate = 0.01  # Augmenté pour accélérer l'apprentissage
    batch_size = 32
    
    print(f"Configuration:")
    print(f"- Nombre de clients: {n_clients}")
    print(f"- Seeds par client: {n_seeds_per_client}")
    print(f"- Features d'entrée: {n_x}")
    print(f"- Unités cachées: {n_a}")
    print(f"- Longueur séquence: {sequence_length}")
    print(f"- Époques: {num_epochs}")
    print(f"- Taux d'apprentissage: {learning_rate}")
    print()
    
    try:
        # Étape 1: Chargement et préparation des données
        print("ÉTAPE 1: Chargement des données")
        print("-" * 40)
        
        clients_data = load_jena_climate_data(
            n_clients=n_clients,
            n_seeds_per_client=n_seeds_per_client,
            n_x=n_x,
            n_y=n_y,
            sequence_length=sequence_length
        )
        
        print(f"Données chargées avec succès pour {len(clients_data)} clients")
        print()
        
        # Étape 2: Entraînement des modèles pour chaque client
        print("ÉTAPE 2: Entraînement des modèles clients")
        print("-" * 40)
        
        parameters_list = []
        training_histories = []
        
        for client_id, client_seeds_data in enumerate(clients_data):
            if len(client_seeds_data) > 0:
                print(f"\nEntraînement du Client {client_id + 1}/{len(clients_data)}")
                print("-" * 30)
                
                # Récupération des données du client
                X_train, Y_train = client_seeds_data[0]
                print(f"Forme des données d'entraînement: {X_train.shape} -> {Y_train.shape}")
                
                # Entraînement du modèle LSTM pour régression
                try:
                    params, params_history, loss_history = train_lstm_regression(
                        X_train, Y_train, n_a, n_x, n_y,
                        num_epochs=num_epochs,
                        learning_rate=learning_rate,
                        seed=client_id + 1
                    )
                    
                    parameters_list.append(params)
                    training_histories.append(loss_history)
                    
                    print(f"Entraînement terminé - Loss finale: {loss_history[-1]:.4f}")
                    
                except Exception as e:
                    print(f"Erreur lors de l'entraînement du client {client_id}: {str(e)}")
                    continue
            else:
                print(f"Client {client_id} n'a pas de données valides")
        
        if not parameters_list:
            raise ValueError("Aucun client n'a pu être entraîné avec succès")
        
        print(f"\nEntraînement terminé pour {len(parameters_list)} clients")
        print()
        
        # Étape 3: Évaluation des performances
        print("ÉTAPE 3: Évaluation des performances")
        print("-" * 40)
        
        client_metrics = evaluate_and_plot_clients(
            clients_data, parameters_list, n_a, n_x, n_y,
            save_path="client_performance_results.png"
        )
        
        # Étape 4: Création du dataset de test global
        print("\nÉTAPE 4: Test sur données globales")
        print("-" * 40)
        
        try:
            X_test, Y_test = create_jena_test_dataset(
                clients_data, batch_size, n_x, n_y, sequence_length
            )
            
            print("Test des modèles sur le dataset global:")
            test_results = []
            
            for client_id, params in enumerate(parameters_list):
                # Test du modèle client sur les données globales
                a0 = np.zeros((n_a, X_test.shape[1]))
                a, y_pred, c, caches = lstm_forward_regression(X_test, a0, params)
                
                # Calcul des métriques
                rmse = compute_rmse(Y_test, y_pred)
                loss = compute_loss(Y_test, y_pred)
                
                test_results.append({
                    'client_id': client_id,
                    'rmse': rmse,
                    'loss': loss
                })
                
                print(f"Client {client_id} sur test global: RMSE = {rmse:.4f}, Loss = {loss:.4f}")
            
            # Meilleur modèle global
            best_client_rmse = min(test_results, key=lambda x: x['rmse'])
            best_client_loss = min(test_results, key=lambda x: x['loss'])
            
            print(f"\nMeilleur modèle (RMSE): Client {best_client_rmse['client_id']} - {best_client_rmse['rmse']:.4f}")
            print(f"Meilleur modèle (Loss): Client {best_client_loss['client_id']} - {best_client_loss['loss']:.4f}")
            
        except Exception as e:
            print(f"Erreur lors du test global: {str(e)}")
        
        # Étape 5: Visualisation des courbes d'entraînement
        print("\nÉTAPE 5: Visualisation des courbes d'entraînement")
        print("-" * 40)
        
        plt.figure(figsize=(12, 6))
        
        for client_id, loss_history in enumerate(training_histories):
            plt.plot(loss_history, label=f'Client {client_id}', linewidth=2)
        
        plt.xlabel('Époques')
        plt.ylabel('Loss')
        plt.title('Évolution de la Loss durant l\'entraînement')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig('training_curves.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        print("Courbes d'entraînement sauvegardées: training_curves.png")
        
        # Résumé final
        print("\n" + "=" * 60)
        print("RÉSUMÉ FINAL")
        print("=" * 60)
        print(f"Nombre de clients entraînés: {len(parameters_list)}")
        print(f"Configuration utilisée: {n_a} unités cachées, {num_epochs} époques")
        print(f"Fichiers générés:")
        print(f"- client_performance_results.png (performances par client)")
        print(f"- training_curves.png (courbes d'entraînement)")
        
        if client_metrics:
            rmse_scores = client_metrics['rmse_scores']
            loss_scores = client_metrics['loss_scores']
            print(f"\nPerformances moyennes:")
            print(f"- RMSE moyen: {np.mean(rmse_scores):.4f} ± {np.std(rmse_scores):.4f}")
            print(f"- Loss moyenne: {np.mean(loss_scores):.4f} ± {np.std(loss_scores):.4f}")
        
        print("\nEntraînement fédéré terminé avec succès!")
        
        return {
            'clients_data': clients_data,
            'parameters_list': parameters_list,
            'client_metrics': client_metrics,
            'training_histories': training_histories
        }
        
    except FileNotFoundError as e:
        print(f"Erreur: {str(e)}")
        print("\nPour résoudre ce problème:")
        print("1. Téléchargez le dataset depuis: https://www.kaggle.com/stytch16/weather-dataset")
        print("2. Placez le fichier 'jena_climate_2009_2016.csv' dans le dossier courant")
        print("3. Ou utilisez: kaggle datasets download -d stytch16/weather-dataset")
        return None
        
    except Exception as e:
        print(f"Erreur inattendue: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # Exécution du programme principal
    results = main()
    
    if results is not None:
        print("\nRésultats disponibles dans la variable 'results':")
        print("- results['clients_data']: données des clients")
        print("- results['parameters_list']: paramètres entraînés")
        print("- results['client_metrics']: métriques d'évaluation")
        print("- results['training_histories']: historiques d'entraînement")

LSTM FÉDÉRÉ - JENA CLIMATE DATASET
Configuration:
- Nombre de clients: 3
- Seeds par client: 1
- Features d'entrée: 8
- Unités cachées: 32
- Longueur séquence: 10
- Époques: 10
- Taux d'apprentissage: 0.01

ÉTAPE 1: Chargement des données
----------------------------------------
Chargement du Jena Climate Dataset
Données brutes: (420551, 15)
Données nettoyées: (420551, 14)
Features sélectionnées: ['p (mbar)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)', 'wd (deg)', 'rh (%)', 'Tdew (degC)', 'VPmax (mbar)']
Création de 3 clients avec 1 seeds chacun
Client 0: 5000 échantillons
  Seed 0: (8, 2000, 10) -> (1, 2000, 1)
Client 1: 5000 échantillons
  Seed 0: (8, 2000, 10) -> (1, 2000, 1)
Client 2: 5000 échantillons
  Seed 0: (8, 2000, 10) -> (1, 2000, 1)
3 clients créés
Données chargées avec succès pour 3 clients

ÉTAPE 2: Entraînement des modèles clients
----------------------------------------

Entraînement du Client 1/3
------------------------------
Forme des données d'entraînement: (8, 20

C:\Users\ikram\AppData\Local\Temp\ipykernel_152012\2917893465.py:117: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
Traceback (most recent call last):
  File "C:\Users\ikram\AppData\Local\Temp\ipykernel_152012\1803955791.py", line 92, in main
    client_metrics = evaluate_and_plot_clients(
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ikram\AppData\Local\Temp\ipykernel_152012\2917893465.py", line 151, in evaluate_and_plot_clients
    plot_client_performance(client_metrics, save_path)
  File "C:\Users\ikram\AppData\Local\Temp\ipykernel_152012\2917893465.py", line 120, in plot_client_performance
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
  File "C:\Users\ikram\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\matplotlib\pyplot.py", line 1243, in savefig
    res = fig.savefig(*ar

Erreur inattendue: Image size of 3740x16826605 pixels is too large. It must be less than 2^23 in each direction.
Error in callback <function flush_figures at 0x000001D8847FDC60> (for post_execute), with arguments args (),kwargs {}:


KeyboardInterrupt: 